# 🚀 Introdução ao Kimi via API

Este notebook apresenta recursos iniciais do **Kimi** (Moonshot AI) consumidos via API, mostrando o quão poderoso pode ser o modelo em tarefas de linguagem, visão, ferramentas e geração estruturada.

**Modelo usado:** `kimi-k3` (ou `kimi-k2-0711-preview`, conforme disponibilidade)

**Documentação:** https://platform.moonshot.cn/docs

## 1. Instalação e configuração

In [1]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install openai

  Using cached openai-3.25.0-py3-none-any.whl.metadata (44 kB)
  Using cached httpx2-2.13.1-py3-none-any.whl.metadata (9.8 kB)
  Using cached jiter-0.17.0-cp314-cp314-win_amd64.whl.metadata (5.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore2-2.13.1-py3-none-any.whl.metadata (26 kB)
Using cached openai-3.25.0-py3-none-any.whl (2.1 MB)
Using cached httpx2-2.13.1-py3-none-any.whl (95 kB)
Using cached httpcore2-2.13.1-py3-none-any.whl (83 kB)
Using cached jiter-0.17.0-cp314-cp314-win_amd64.whl (227 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

   ---------------- ----------------------- 2/5 [httpcore2]
   ------------------------ --------------- 3/5 [httpx2]
   ------------------------ --------------- 3/5 [httpx2]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [ope

In [5]:
# Instale a biblioteca oficial (OpenAI-compatible)
!pip install -q openai python-dotenv

In [19]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("KIMI_API_KEY"),
    base_url="https://api.moonshot.cn/v1"
)

MODEL = "kimi-k2.6"

## 2. Chat completion simples

O básico: envie uma lista de mensagens e receba uma resposta.

In [23]:
def chat(txt, temp=1.0):
    response = client.chat.completions.create(
        model=MODEL,
        messages=txt,
        temperature=temp
    )
    return response.choices[0].message.content
    
msg= [
        {"role": "system", "content": "Você é um assistente tecnico especialista em Python"},
        {"role": "user", "content": "Explique de forma mais resumida possivel o que são closures em Python, cum um exemplo pratico."}
]

print(chat(msg))

**Closure** é uma função interna que "lembra" as variáveis da função externa, mesmo depois que a externa já terminou de executar.

```python
def multiplicador(n):
    def func(x):
        return x * n  # 'n' veio da função externa
    return func

dobro = multiplicador(2)
triplo = multiplicador(3)

print(dobro(5))   # 10
print(triplo(5))  # 15
```

**O que acontece:** `dobro` e `triplo` são closures. Cada uma "guardou" o valor de `n` (2 e 3) e o usa depois, mesmo sem `multiplicador()` estar mais em execução.


## 3. Respostas em streaming

O streaming permite mostrar texto aos poucos, ideal para interfaces interativas.

In [20]:
msg= [
        {"role": "system", "content": "Você é um assistente tecnico especialista em Python"},
        {"role": "user", "content": "Explique de forma mais resumida possivel o que são closures em Python, cum um exemplo pratico."}
]
stream = client.chat.completions.create(
    model=MODEL,
    messages=msg,
    stream=True,
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

**Closure** é uma função interna que "lembra" das variáveis da função externa, mesmo depois que ela já terminou de executar.

```python
def multiplicador(fator):
    def calcular(numero):
        return numero * fator  # lembra de 'fator' mesmo depois que 'multiplicador' acabou
    return calcular

dobro = multiplicador(2)
triplo = multiplicador(3)

print(dobro(5))   # 10
print(triplo(5))  # 15
```

A função `calcular` "fecha" (*closure*) sobre o valor de `fator`, permitindo reusá-lo depois.

## 4. System prompt e controle de comportamento

O system prompt define a persona, formato e restrições do modelo.

In [24]:
system_prompt = """
Você é um tradutor técnico sênior. Regras:
1. Traduza do português para o inglês técnico.
2. Mantenha termos de código e APIs em inglês.
3. Responda APENAS com a tradução, sem explicações.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "O método __init__ é o construtor da classe e recebe self como primeiro argumento."}
]

print(chat(messages))

The __init__ method is the class constructor and receives self as its first argument.


## 5. Function Calling (tools)

O modelo pode decidir chamar funções externas para obter dados em tempo real. Aqui simulamos uma consulta de previsão do tempo.

In [25]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Obtém a previsão do tempo para uma cidade.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Nome da cidade"},
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Unidade de temperatura"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

messages = [
    {"role": "user", "content": "Qual a temperatura em São Paulo agora?"}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
print(message)

ChatCompletionMessage(content='Vou consultar a temperatura atual em São Paulo para você.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='get_weather_0_02f7fd32', function=Function(arguments='{"city": "São Paulo"}', name='get_weather'), type='function', index=0)], reasoning_content='O usuário está perguntando sobre a temperatura atual em São Paulo. Eu tenho a função get_weather disponível para obter a previsão do tempo para uma cidade. Preciso chamar essa função com o nome da cidade "São Paulo".\n\nNão há especificação de unidade de temperatura, então posso usar o padrão ou deixar como está. A unidade padrão provavelmente é celsius, o que faz sentido para uma pergunta em português (Brasil).\n\nVou chamar a função get_weather com city="São Paulo".')


In [26]:
# Simula o resultado da função e devolve ao modelo
def get_weather(city, unit="celsius"):
    return {"city": city, "temperature": 22, "unit": unit, "condition": "parcialmente nublado"}

if message.tool_calls:
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = eval(tool_call.function.arguments)  # em produção, use json.loads
    
    if function_name == "get_weather":
        result = get_weather(**arguments)
    
    messages.append({
        "role": "assistant",
        "content": None,
        "tool_calls": [tool_call.model_dump()]
    })
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(result)
    })
    
    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )
    print(final_response.choices[0].message.content)

A temperatura atual em São Paulo é de **22°C** e o céu está **parcialmente nublado**.


## 6. JSON Mode / saída estruturada

Force o modelo a responder em JSON válido, útil para integração com código.

In [7]:
import json

messages = [
    {"role": "system", "content": "Você é um analisador de produtos. Responda SEMPRE em JSON válido."},
    {
        "role": "user",
        "content": """
Extraia os pontos positivos e negativos do produto descrito abaixo.

Produto: Smartphone KimiPhone Pro
"A câmera é excelente em pouca luz, mas a bateria dura pouco mais de meio dia com uso intenso. O desempenho é fluido e a tela tem cores vibrantes."

Formato esperado:
{
  "produto": "string",
  "pontos_positivos": ["string"],
  "pontos_negativos": ["string"],
  "nota_geral": number
}
"""
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    response_format={"type": "json_object"}
)

data = json.loads(response.choices[0].message.content)
print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "produto": "Smartphone KimiPhone Pro",
  "pontos_positivos": [
    "Câmera excelente em pouca luz",
    "Desempenho fluido",
    "Tela com cores vibrantes"
  ],
  "pontos_negativos": [
    "Bateria dura pouco mais de meio dia com uso intenso"
  ],
  "nota_geral": 3.8
}


## 8. Conversa multi-turn com memória

Mantenha o histórico de mensagens para criar diálogos coerentes.

In [27]:
conversation = [
    {"role": "system", "content": "Você é um tutor de programação paciente. e dá respostas resumidas objetivas e diretas"}
]

def ask(question):
    conversation.append({"role": "user", "content": question})
    answer = chat(conversation)
    conversation.append({"role": "assistant", "content": answer})
    return answer

print("Turno 1:", ask("O que é uma API REST?"))
print("\nTurno 2:", ask("E qual a diferença para GraphQL?"))
print("\nTurno 3:", ask("Me dê um exemplo de quando escolher cada uma?"))

Turno 1: **API REST** é uma interface que permite sistemas se comunicarem pela internet usando o protocolo HTTP.

**Princípios chave:**

*   **Recursos:** cada entidade (usuário, pedido, produto) tem uma URL única (`/usuarios/123`)
*   **Métodos HTTP:** ações são feitas pelos verbos:
    *   `GET` → ler
    *   `POST` → criar
    *   `PUT/PATCH` → atualizar
    *   `DELETE` → remover
*   **Stateless:** cada requisição é independente; o servidor não guarda estado do cliente entre chamadas
*   **Formato:** geralmente troca dados em JSON

**Exemplo prático:**
`GET https://api.exemplo.com/produtos` → retorna a lista de produtos.

Em resumo: é um padrão simples e padronizado para integrar aplicações via web.

Turno 2: **REST** trabalha com **múltiplos endpoints** fixos; **GraphQL** usa **um único endpoint** onde o cliente define exatamente quais dados quer.

|  | REST | GraphQL |
|---|---|---|
| **Requisição** | `GET /usuarios/1`<br>`GET /usuarios/1/pedidos` | Uma query no endpoint `/graphq

## 9. Chain-of-Thought: resolvendo problemas passo a passo

Incentive o modelo a mostrar o raciocínio antes da resposta final.

In [29]:
messages = [
    {
        "role": "user",
        "content": """
Resolva passo a passo:
Um trem parte de A para B a 60 km/h. Outro parte de B para A a 80 km/h.
A distância entre A e B é 280 km. Em quanto tempo eles se encontram?
Mostre o raciocínio completo antes da resposta final.
"""
    }
]

print(chat(messages, temp=1.0))

**Raciocínio completo:**

**1. Identificação dos dados do problema**
*   Trem 1 (saindo de A): velocidade $v_1 = 60$ km/h
*   Trem 2 (saindo de B): velocidade $v_2 = 80$ km/h
*   Distância entre as cidades: $D = 280$ km
*   Tempo decorrido até o encontro: $t$ (incógnita, em horas)

**2. Princípio fundamental**
Como os trens partem ao mesmo tempo e viajam em sentidos opostos um em direção ao outro, no exato instante em que se cruzarem, a **soma das distâncias percorridas por ambos será igual à distância total** entre A e B.

*   Distância percorrida pelo trem 1: $d_1 = v_1 \cdot t = 60t$
*   Distância percorrida pelo trem 2: $d_2 = v_2 \cdot t = 80t$

**3. Montagem da equação**
$$d_1 + d_2 = D$$
$$60t + 80t = 280$$

**4. Resolução**
Somando os termos semelhantes:
$$140t = 280$$

Isolando $t$:
$$t = \frac{280}{140}$$
$$t = 2$$

**5. Verificação (conferindo o resultado)**
*   Trem 1 em 2 horas percorre: $60 \times 2 = 120$ km
*   Trem 2 em 2 horas percorre: $80 \times 2 = 160$ km
*   Soma

## 11. Análise de tokens e custo estimado

A API retorna informações de uso, permitindo estimar custos.

In [30]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Resuma a Revolução Industrial em 3 frases."}]
)

usage = response.usage
print(f"Prompt tokens: {usage.prompt_tokens}")
print(f"Completion tokens: {usage.completion_tokens}")
print(f"Total tokens: {usage.total_tokens}")

Prompt tokens: 20
Completion tokens: 665
Total tokens: 685


## 12. Requisições assíncronas (opcional)

Use a versão assíncrona do cliente para chamadas concorrentes.

In [33]:
import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(
    api_key=os.getenv("KIMI_API_KEY"),
    base_url="https://api.moonshot.cn/v1"
)
MODEL2="kimi-k2.7-code"
async def perguntar(assunto):
    r = await async_client.chat.completions.create(
        model=MODEL2,
        messages=[{"role": "user", "content": f"Explique {assunto} em uma frase."}]
    )
    return r.choices[0].message.content

resultados = await asyncio.gather(
    perguntar("machine learning"),
    perguntar("blockchain"),
    perguntar("computação em nuvem")
)

for r in resultados:
    print("-", r)

RateLimitError: Error code: 429 - {'error': {'message': 'Your account org-d7873c6a73f645b8aa9b32785042abaa<ak-fd4nydghswci11gfgek1> request reached max organization concurrency: 1, please try again after 1 seconds', 'type': 'rate_limit_reached_error'}}

## Próximos passos

- Experimente prompts mais complexos para seu domínio.
- Combine function calling com JSON mode para agents.
- Implemente RAG (Retrieval-Augmented Generation) usando embeddings.
- Avalie diferentes valores de `temperature` e `top_p`.
- Consulte a documentação oficial para limites de contexto e preços.